# 3D Reconstruction Pipeline using COLMAP and OpenMVS

This notebook implements a complete photogrammetry pipeline to reconstruct 3D models from images using:
- **COLMAP**: For Structure-from-Motion (SfM) and Multi-View Stereo (MVS)
- **OpenMVS**: For dense reconstruction and mesh generation

## Pipeline Steps:
1. Setup and Configuration
2. Feature Extraction
3. Feature Matching
4. Sparse Reconstruction
5. Dense Reconstruction
6. Mesh Generation
7. Texture Mapping
8. Export to GLB format

## 1. Import Required Libraries

In [ ]:
import os
import sys
import subprocess
import shutil
from pathlib import Path
import logging
from datetime import datetime

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("Libraries imported successfully!")

## 2. Configuration and Path Setup

In [ ]:
# Configuration
class Config:
    # Base paths
    PROJECT_ROOT = Path("/home/bablu/Bablu/Works/Projects/The_Digital_Plate/3D_models")
    INPUT_IMAGES = PROJECT_ROOT / "dish_images_original/samosa"
    OUTPUT_BASE = PROJECT_ROOT / "dish_models/samosa"
    
    # COLMAP workspace
    COLMAP_WORKSPACE = OUTPUT_BASE / "colmap_workspace"
    COLMAP_DATABASE = COLMAP_WORKSPACE / "database.db"
    COLMAP_IMAGES = COLMAP_WORKSPACE / "images"
    COLMAP_SPARSE = COLMAP_WORKSPACE / "sparse"
    COLMAP_DENSE = COLMAP_WORKSPACE / "dense"
    
    # OpenMVS workspace
    OPENMVS_WORKSPACE = OUTPUT_BASE / "openmvs_workspace"
    
    # Final output
    FINAL_OUTPUT = OUTPUT_BASE / "final_model"
    
    # COLMAP settings
    CAMERA_MODEL = "SIMPLE_RADIAL"  # or "OPENCV", "SIMPLE_PINHOLE", etc.
    MATCHING_METHOD = "exhaustive"  # or "sequential", "vocab_tree"
    
    # Quality settings
    QUALITY = "high"  # "low", "medium", "high", "extreme"

# Create all necessary directories
def setup_directories():
    """Create all required directories for the pipeline"""
    directories = [
        Config.OUTPUT_BASE,
        Config.COLMAP_WORKSPACE,
        Config.COLMAP_IMAGES,
        Config.COLMAP_SPARSE,
        Config.COLMAP_DENSE,
        Config.OPENMVS_WORKSPACE,
        Config.FINAL_OUTPUT
    ]
    
    for directory in directories:
        directory.mkdir(parents=True, exist_ok=True)
        logger.info(f"Directory ready: {directory}")
    
    return True

# Setup directories
if setup_directories():
    logger.info("✓ All directories created successfully!")
    logger.info(f"Input images: {len(list(Config.INPUT_IMAGES.glob('*.jpg')))} files found")
else:
    logger.error("✗ Failed to setup directories")

## 3. Utility Functions

In [ ]:
def run_command(command, description="Running command", check=True):
    """
    Execute a shell command with logging
    
    Args:
        command (list): Command and arguments as a list
        description (str): Description of the command
        check (bool): Whether to raise exception on failure
    
    Returns:
        subprocess.CompletedProcess: Result of the command
    """
    logger.info(f"{'='*60}")
    logger.info(f"{description}")
    logger.info(f"Command: {' '.join(command)}")
    logger.info(f"{'='*60}")
    
    try:
        result = subprocess.run(
            command,
            check=check,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )
        
        if result.stdout:
            logger.info(f"Output:\n{result.stdout}")
        
        logger.info(f"✓ {description} - COMPLETED")
        return result
        
    except subprocess.CalledProcessError as e:
        logger.error(f"✗ {description} - FAILED")
        logger.error(f"Error: {e.stderr}")
        raise
    except FileNotFoundError:
        logger.error(f"✗ Command not found. Please ensure COLMAP/OpenMVS is installed.")
        raise

def check_dependencies():
    """Check if required software is installed"""
    dependencies = {
        'colmap': ['colmap', '--version'],
        'openmvs': ['DensifyPointCloud', '--version']
    }
    
    missing = []
    for name, command in dependencies.items():
        try:
            subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
            logger.info(f"✓ {name.upper()} is installed")
        except (subprocess.CalledProcessError, FileNotFoundError):
            logger.warning(f"✗ {name.upper()} not found or not working")
            missing.append(name)
    
    if missing:
        logger.warning(f"Missing dependencies: {', '.join(missing)}")
        logger.warning("The pipeline may fail. Please install missing dependencies.")
    else:
        logger.info("✓ All dependencies are available!")
    
    return len(missing) == 0

# Check dependencies
check_dependencies()

## 4. Prepare Images for COLMAP

In [ ]:
def prepare_images():
    """Copy images to COLMAP workspace"""
    logger.info("Preparing images for COLMAP...")
    
    # Clear existing images in COLMAP workspace
    if Config.COLMAP_IMAGES.exists():
        shutil.rmtree(Config.COLMAP_IMAGES)
    Config.COLMAP_IMAGES.mkdir(parents=True, exist_ok=True)
    
    # Copy images
    image_files = list(Config.INPUT_IMAGES.glob("*.jpg")) + \
                  list(Config.INPUT_IMAGES.glob("*.JPG")) + \
                  list(Config.INPUT_IMAGES.glob("*.png")) + \
                  list(Config.INPUT_IMAGES.glob("*.PNG"))
    
    if not image_files:
        raise FileNotFoundError(f"No images found in {Config.INPUT_IMAGES}")
    
    logger.info(f"Found {len(image_files)} images")
    
    for img in image_files:
        shutil.copy2(img, Config.COLMAP_IMAGES / img.name)
    
    logger.info(f"✓ Copied {len(image_files)} images to COLMAP workspace")
    return len(image_files)

# Prepare images
num_images = prepare_images()
print(f"\nReady to process {num_images} images")

## 5. COLMAP: Feature Extraction

In [ ]:
def extract_features():
    """Extract features from images using COLMAP"""
    command = [
        "colmap", "feature_extractor",
        "--database_path", str(Config.COLMAP_DATABASE),
        "--image_path", str(Config.COLMAP_IMAGES),
        "--ImageReader.camera_model", Config.CAMERA_MODEL,
        "--ImageReader.single_camera", "1",
        "--SiftExtraction.use_gpu", "1"  # Set to 0 if no GPU available
    ]
    
    run_command(
        command,
        description="COLMAP Feature Extraction"
    )
    
    logger.info("✓ Feature extraction completed!")

# Run feature extraction
extract_features()

## 6. COLMAP: Feature Matching

In [ ]:
def match_features():
    """Match features between images using COLMAP"""
    command = [
        "colmap", f"{Config.MATCHING_METHOD}_matcher",
        "--database_path", str(Config.COLMAP_DATABASE),
        "--SiftMatching.use_gpu", "1"  # Set to 0 if no GPU available
    ]
    
    run_command(
        command,
        description="COLMAP Feature Matching"
    )
    
    logger.info("✓ Feature matching completed!")

# Run feature matching
match_features()

## 7. COLMAP: Sparse Reconstruction (SfM)

In [ ]:
def sparse_reconstruction():
    """Perform sparse reconstruction using COLMAP"""
    # Create sparse output directory
    sparse_output = Config.COLMAP_SPARSE / "0"
    sparse_output.mkdir(parents=True, exist_ok=True)
    
    command = [
        "colmap", "mapper",
        "--database_path", str(Config.COLMAP_DATABASE),
        "--image_path", str(Config.COLMAP_IMAGES),
        "--output_path", str(Config.COLMAP_SPARSE)
    ]
    
    run_command(
        command,
        description="COLMAP Sparse Reconstruction (Structure-from-Motion)"
    )
    
    logger.info("✓ Sparse reconstruction completed!")
    
    # Check if reconstruction was successful
    model_files = list((Config.COLMAP_SPARSE / "0").glob("*"))
    if model_files:
        logger.info(f"Generated {len(model_files)} model files")
    else:
        logger.warning("No model files generated. Check if reconstruction was successful.")

# Run sparse reconstruction
sparse_reconstruction()

## 8. COLMAP: Image Undistortion

In [ ]:
def undistort_images():
    """Undistort images for dense reconstruction"""
    command = [
        "colmap", "image_undistorter",
        "--image_path", str(Config.COLMAP_IMAGES),
        "--input_path", str(Config.COLMAP_SPARSE / "0"),
        "--output_path", str(Config.COLMAP_DENSE),
        "--output_type", "COLMAP"
    ]
    
    run_command(
        command,
        description="COLMAP Image Undistortion"
    )
    
    logger.info("✓ Image undistortion completed!")

# Run image undistortion
undistort_images()

## 9. COLMAP: Dense Stereo Reconstruction

In [ ]:
def dense_stereo():
    """Perform dense stereo reconstruction using COLMAP"""
    command = [
        "colmap", "patch_match_stereo",
        "--workspace_path", str(Config.COLMAP_DENSE),
        "--workspace_format", "COLMAP",
        "--PatchMatchStereo.geom_consistency", "true"
    ]
    
    run_command(
        command,
        description="COLMAP Dense Stereo Reconstruction"
    )
    
    logger.info("✓ Dense stereo reconstruction completed!")

# Run dense stereo reconstruction
dense_stereo()

## 10. COLMAP: Stereo Fusion (Point Cloud Generation)

In [ ]:
def stereo_fusion():
    """Fuse stereo depth maps into a point cloud"""
    output_ply = Config.COLMAP_DENSE / "fused.ply"
    
    command = [
        "colmap", "stereo_fusion",
        "--workspace_path", str(Config.COLMAP_DENSE),
        "--workspace_format", "COLMAP",
        "--input_type", "geometric",
        "--output_path", str(output_ply)
    ]
    
    run_command(
        command,
        description="COLMAP Stereo Fusion"
    )
    
    logger.info(f"✓ Stereo fusion completed! Point cloud saved to: {output_ply}")
    return output_ply

# Run stereo fusion
point_cloud_file = stereo_fusion()

## 11. Convert COLMAP to OpenMVS Format

In [ ]:
def convert_to_openmvs():
    """Convert COLMAP model to OpenMVS format"""
    output_mvs = Config.OPENMVS_WORKSPACE / "scene.mvs"
    
    command = [
        "InterfaceCOLMAP",
        "-i", str(Config.COLMAP_DENSE),
        "-o", str(output_mvs),
        "--image-folder", str(Config.COLMAP_DENSE / "images")
    ]
    
    run_command(
        command,
        description="Converting COLMAP to OpenMVS Format"
    )
    
    logger.info(f"✓ Conversion completed! OpenMVS scene saved to: {output_mvs}")
    return output_mvs

# Convert to OpenMVS format
mvs_scene = convert_to_openmvs()

## 12. OpenMVS: Densify Point Cloud

In [ ]:
def densify_point_cloud():
    """Densify the point cloud using OpenMVS"""
    input_mvs = Config.OPENMVS_WORKSPACE / "scene.mvs"
    output_mvs = Config.OPENMVS_WORKSPACE / "scene_dense.mvs"
    
    command = [
        "DensifyPointCloud",
        "-i", str(input_mvs),
        "-o", str(output_mvs),
        "--resolution-level", "1",  # 0=highest quality, 1=high, 2=medium
        "-w", str(Config.OPENMVS_WORKSPACE)
    ]
    
    run_command(
        command,
        description="OpenMVS Densify Point Cloud"
    )
    
    logger.info(f"✓ Point cloud densification completed!")
    return output_mvs

# Densify point cloud
dense_mvs = densify_point_cloud()

## 13. OpenMVS: Reconstruct Mesh

In [ ]:
def reconstruct_mesh():
    """Reconstruct mesh from dense point cloud using OpenMVS"""
    input_mvs = Config.OPENMVS_WORKSPACE / "scene_dense.mvs"
    output_mvs = Config.OPENMVS_WORKSPACE / "scene_mesh.mvs"
    
    command = [
        "ReconstructMesh",
        "-i", str(input_mvs),
        "-o", str(output_mvs),
        "-w", str(Config.OPENMVS_WORKSPACE)
    ]
    
    run_command(
        command,
        description="OpenMVS Mesh Reconstruction"
    )
    
    logger.info(f"✓ Mesh reconstruction completed!")
    return output_mvs

# Reconstruct mesh
mesh_mvs = reconstruct_mesh()

## 14. OpenMVS: Refine Mesh

In [ ]:
def refine_mesh():
    """Refine the reconstructed mesh using OpenMVS"""
    input_mvs = Config.OPENMVS_WORKSPACE / "scene_mesh.mvs"
    output_mvs = Config.OPENMVS_WORKSPACE / "scene_mesh_refine.mvs"
    
    command = [
        "RefineMesh",
        "-i", str(input_mvs),
        "-o", str(output_mvs),
        "--resolution-level", "1",
        "-w", str(Config.OPENMVS_WORKSPACE)
    ]
    
    run_command(
        command,
        description="OpenMVS Mesh Refinement"
    )
    
    logger.info(f"✓ Mesh refinement completed!")
    return output_mvs

# Refine mesh
refined_mesh_mvs = refine_mesh()

## 15. OpenMVS: Texture Mesh

In [ ]:
def texture_mesh():
    """Apply texture to the mesh using OpenMVS"""
    input_mvs = Config.OPENMVS_WORKSPACE / "scene_mesh_refine.mvs"
    output_mvs = Config.OPENMVS_WORKSPACE / "scene_mesh_texture.mvs"
    
    command = [
        "TextureMesh",
        "-i", str(input_mvs),
        "-o", str(output_mvs),
        "--resolution-level", "1",
        "-w", str(Config.OPENMVS_WORKSPACE)
    ]
    
    run_command(
        command,
        description="OpenMVS Texture Mapping"
    )
    
    logger.info(f"✓ Texture mapping completed!")
    
    # Check for output files
    output_obj = Config.OPENMVS_WORKSPACE / "scene_mesh_texture.obj"
    output_ply = Config.OPENMVS_WORKSPACE / "scene_mesh_texture.ply"
    
    if output_obj.exists():
        logger.info(f"✓ Textured mesh exported to: {output_obj}")
        return output_obj
    elif output_ply.exists():
        logger.info(f"✓ Textured mesh exported to: {output_ply}")
        return output_ply
    else:
        logger.warning("No mesh output file found")
        return output_mvs

# Apply texture to mesh
textured_mesh = texture_mesh()

## 16. Convert to GLB Format (Optional)

In [ ]:
def convert_to_glb(input_mesh):
    """
    Convert mesh to GLB format using trimesh or obj2gltf
    This requires additional libraries like trimesh or obj2gltf
    """
    try:
        import trimesh
        
        output_glb = Config.FINAL_OUTPUT / "samosa_model.glb"
        
        logger.info(f"Loading mesh from: {input_mesh}")
        
        # Load the mesh
        if str(input_mesh).endswith('.obj'):
            mesh = trimesh.load(str(input_mesh), force='mesh')
        elif str(input_mesh).endswith('.ply'):
            mesh = trimesh.load(str(input_mesh), force='mesh')
        else:
            logger.warning(f"Unsupported format: {input_mesh}")
            return None
        
        # Export to GLB
        mesh.export(str(output_glb), file_type='glb')
        logger.info(f"✓ GLB file saved to: {output_glb}")
        
        return output_glb
        
    except ImportError:
        logger.warning("trimesh library not installed. Install it with: pip install trimesh")
        logger.info("Alternative: Use Blender or online converter to convert OBJ/PLY to GLB")
        
        # Copy the final mesh to output folder
        if input_mesh.exists():
            output_mesh = Config.FINAL_OUTPUT / input_mesh.name
            shutil.copy2(input_mesh, output_mesh)
            logger.info(f"✓ Final mesh copied to: {output_mesh}")
            return output_mesh
        
        return None
    
    except Exception as e:
        logger.error(f"Error during GLB conversion: {e}")
        
        # Copy the final mesh to output folder as fallback
        if input_mesh.exists():
            output_mesh = Config.FINAL_OUTPUT / input_mesh.name
            shutil.copy2(input_mesh, output_mesh)
            logger.info(f"✓ Final mesh copied to: {output_mesh}")
            return output_mesh
        
        return None

# Convert to GLB
if textured_mesh and Path(textured_mesh).exists():
    final_model = convert_to_glb(textured_mesh)
else:
    logger.warning("No textured mesh found to convert")

## 17. Complete Pipeline Execution (Run All Steps)

In [ ]:
def run_complete_pipeline():
    """
    Execute the complete photogrammetry pipeline
    """
    try:
        start_time = datetime.now()
        logger.info("="*80)
        logger.info("STARTING COMPLETE 3D RECONSTRUCTION PIPELINE")
        logger.info("="*80)
        
        # Step 1: Setup
        logger.info("\n[1/15] Setting up directories...")
        setup_directories()
        
        # Step 2: Check dependencies
        logger.info("\n[2/15] Checking dependencies...")
        check_dependencies()
        
        # Step 3: Prepare images
        logger.info("\n[3/15] Preparing images...")
        num_images = prepare_images()
        
        # Step 4: Feature extraction
        logger.info("\n[4/15] Extracting features...")
        extract_features()
        
        # Step 5: Feature matching
        logger.info("\n[5/15] Matching features...")
        match_features()
        
        # Step 6: Sparse reconstruction
        logger.info("\n[6/15] Performing sparse reconstruction...")
        sparse_reconstruction()
        
        # Step 7: Image undistortion
        logger.info("\n[7/15] Undistorting images...")
        undistort_images()
        
        # Step 8: Dense stereo
        logger.info("\n[8/15] Computing dense stereo...")
        dense_stereo()
        
        # Step 9: Stereo fusion
        logger.info("\n[9/15] Fusing stereo depth maps...")
        point_cloud = stereo_fusion()
        
        # Step 10: Convert to OpenMVS
        logger.info("\n[10/15] Converting to OpenMVS format...")
        mvs_scene = convert_to_openmvs()
        
        # Step 11: Densify point cloud
        logger.info("\n[11/15] Densifying point cloud...")
        dense_scene = densify_point_cloud()
        
        # Step 12: Reconstruct mesh
        logger.info("\n[12/15] Reconstructing mesh...")
        mesh_scene = reconstruct_mesh()
        
        # Step 13: Refine mesh
        logger.info("\n[13/15] Refining mesh...")
        refined_mesh = refine_mesh()
        
        # Step 14: Texture mesh
        logger.info("\n[14/15] Applying texture...")
        textured = texture_mesh()
        
        # Step 15: Convert to GLB
        logger.info("\n[15/15] Converting to GLB format...")
        final = convert_to_glb(textured)
        
        # Summary
        end_time = datetime.now()
        duration = end_time - start_time
        
        logger.info("="*80)
        logger.info("PIPELINE COMPLETED SUCCESSFULLY!")
        logger.info("="*80)
        logger.info(f"Total time: {duration}")
        logger.info(f"Input images: {num_images}")
        logger.info(f"Final model: {final}")
        logger.info("="*80)
        
        return final
        
    except Exception as e:
        logger.error("="*80)
        logger.error("PIPELINE FAILED!")
        logger.error("="*80)
        logger.error(f"Error: {str(e)}")
        logger.error("Check the logs above for details")
        raise

# Uncomment the line below to run the complete pipeline
# final_model = run_complete_pipeline()

## Additional Notes and Tips

### Installation Requirements

Before running this notebook, ensure you have the following installed:

1. **COLMAP**
   ```bash
   # Ubuntu/Debian
   sudo apt-get install colmap
   
   # Or build from source for latest version
   # See: https://colmap.github.io/install.html
   ```

2. **OpenMVS**
   ```bash
   # Ubuntu/Debian
   sudo apt-get install openmvs
   
   # Or build from source
   # See: https://github.com/cdcseacave/openMVS
   ```

3. **Python Libraries**
   ```bash
   pip install trimesh numpy pillow
   ```

### GPU Acceleration

- Set `--SiftExtraction.use_gpu 0` and `--SiftMatching.use_gpu 0` if no GPU available
- GPU significantly speeds up feature extraction and matching

### Quality Settings

Adjust quality in the Config class:
- `QUALITY = "low"`: Fast processing, lower quality
- `QUALITY = "medium"`: Balanced
- `QUALITY = "high"`: Better quality, slower
- `QUALITY = "extreme"`: Best quality, very slow

### Troubleshooting

1. **Out of memory**: Reduce resolution-level (increase number)
2. **Poor reconstruction**: Ensure good image overlap and lighting
3. **Missing features**: Check camera movement covers all angles
4. **Failed matching**: Try different MATCHING_METHOD (sequential/exhaustive)

### Output Files

- COLMAP sparse model: `colmap_workspace/sparse/0/`
- COLMAP dense point cloud: `colmap_workspace/dense/fused.ply`
- OpenMVS final mesh: `openmvs_workspace/scene_mesh_texture.obj`
- Final GLB model: `final_model/samosa_model.glb`